# SQL practice — orders / customers / products

Тренажёр на sqlite3 (встроен в Python, ничего дополнительно ставить не нужно).

Раньше слепая зона была именно в JOIN между таблицами (путал `orders`/`customers`,
терял поля не из той таблицы) и в COUNT vs SUM. Схема здесь специально на 3 таблицы,
чтобы прицельно тренировать это.

**Схема:**

```
customers(customer_id, name, city, signup_date)
products(product_id, name, category, price)
orders(order_id, customer_id, order_date, status)
order_items(order_item_id, order_id, product_id, quantity)
```

Сумма заказа не хранится напрямую — она считается через `order_items.quantity * products.price`.
Это специально: заставляет каждый раз думать, откуда берётся значение, а не просто читать колонку.


## Setup — выполнить один раз в начале сессии

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    signup_date TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name TEXT,
    category TEXT,
    price REAL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    status TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

conn.commit()
print("Схема создана")


In [ ]:
cur.executescript("""
INSERT INTO customers (customer_id, name, city, signup_date) VALUES
    (1, 'Alma Zharkynbekova', 'Almaty', '2025-11-02'),
    (2, 'Ruslan Dzhaksybekov', 'Astana', '2026-01-15'),
    (3, 'Dana Serikova', 'Shymkent', '2026-02-20'),
    (4, 'Yerlan Mukanov', 'Almaty', '2026-03-05'),
    (5, 'Aigerim Tulegenova', 'Astana', '2026-04-11');

INSERT INTO products (product_id, name, category, price) VALUES
    (1, 'Wireless Mouse', 'Electronics', 4500),
    (2, 'Mechanical Keyboard', 'Electronics', 21000),
    (3, 'Notebook A5', 'Office', 900),
    (4, 'Desk Lamp', 'Home', 6500),
    (5, 'USB-C Cable', 'Electronics', 2500);

INSERT INTO orders (order_id, customer_id, order_date, status) VALUES
    (101, 1, '2026-07-02', 'completed'),
    (102, 2, '2026-07-05', 'completed'),
    (103, 1, '2026-07-20', 'completed'),
    (104, 3, '2026-08-01', 'cancelled'),
    (105, 2, '2026-08-10', 'completed'),
    (106, 4, '2026-08-15', 'completed'),
    (107, 1, '2026-08-22', 'completed');

INSERT INTO order_items (order_item_id, order_id, product_id, quantity) VALUES
    (1, 101, 1, 2),
    (2, 101, 3, 5),
    (3, 102, 2, 1),
    (4, 103, 5, 3),
    (5, 104, 4, 1),
    (6, 105, 1, 1),
    (7, 105, 5, 2),
    (8, 106, 2, 1),
    (9, 106, 4, 2),
    (10, 107, 3, 10);
""")

conn.commit()
print("Данные загружены")


### Заметка: customer 5 (Aigerim) без единого заказа — специально, для anti-join задачи.

In [ ]:
def q(sql):
    """Выполнить SELECT и красиво распечатать результат."""
    cur.execute(sql)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    widths = [max(len(str(c)), *(len(str(r[i])) for r in rows)) if rows else len(str(c)) for i, c in enumerate(cols)]
    print(" | ".join(c.ljust(w) for c, w in zip(cols, widths)))
    print("-+-".join("-" * w for w in widths))
    for r in rows:
        print(" | ".join(str(v).ljust(w) for v, w in zip(r, widths)))
    print(f"\n({len(rows)} rows)")


---
## Упражнение 1 — разминка (1 таблица)

Все заказы со статусом `completed` за август 2026.

In [ ]:
q("""
-- твой запрос
""")


## Упражнение 2 — JOIN (2 таблицы)

Имя клиента и дата заказа для всех заказов из Алматы (`city = 'Almaty'`).

In [ ]:
q("""
-- твой запрос
""")


## Упражнение 3 — JOIN (3 таблицы) + правильная агрегатная функция

Сумма каждого заказа (`order_id`) — то есть `SUM(quantity * price)` по товарам в этом заказе,
только для заказов со статусом `completed`.

Подсказка на твою старую ошибку: тут нужен `SUM`, не `COUNT` — считаем деньги, не строки.

In [ ]:
q("""
-- твой запрос
""")


## Упражнение 4 — GROUP BY + JOIN

Общая сумма покупок по городам (`customers.city`), только `completed` заказы.
Отсортировать по убыванию суммы.

In [ ]:
q("""
-- твой запрос
""")


## Упражнение 5 — анти-джойн

Клиенты, у которых вообще нет ни одного заказа (не только completed — вообще).

Это тот паттерн, который в прошлый раз ты вспомнил сам без подсказки — проверим, что закрепилось.

In [ ]:
q("""
-- твой запрос
""")


## Упражнение 6 — последний заказ по клиенту

Для каждого клиента — дата его самого последнего заказа (`MAX(order_date)`).
Только клиенты, у которых есть хотя бы один заказ.

Подсказка на старую ошибку: `DISTINCT` и `MAX()` нельзя мешать без `GROUP BY` — тут нужен `GROUP BY`.

In [ ]:
q("""
-- твой запрос
""")


---
## Ответы (не подглядывать, пока не попробовал сам)

<details>
<summary>Раскрыть решения</summary>

**1.**
```sql
SELECT * FROM orders
WHERE status = 'completed'
  AND order_date >= '2026-08-01' AND order_date < '2026-09-01';
```

**2.**
```sql
SELECT c.name, o.order_date
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE c.city = 'Almaty';
```

**3.**
```sql
SELECT o.order_id, SUM(oi.quantity * p.price) AS order_total
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.status = 'completed'
GROUP BY o.order_id;
```

**4.**
```sql
SELECT c.city, SUM(oi.quantity * p.price) AS total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.status = 'completed'
GROUP BY c.city
ORDER BY total DESC;
```

**5.**
```sql
SELECT c.*
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL;
```

**6.**
```sql
SELECT customer_id, MAX(order_date) AS last_order
FROM orders
GROUP BY customer_id;
```

</details>
